In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["Price"] = data.target
X = df[["MedInc"]]
y = df["Price"]

print("Dataset Loaded Successfully")
display(df[["MedInc", "Price"]].head())
print("Dataset Shape:", df.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
degrees = [1, 2, 3, 5, 10]
results = []

for degree in degrees:
    polynomial = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = polynomial.fit_transform(X_train)
    X_test_poly = polynomial.transform(X_test)
    model = LinearRegression()

    start_time = time.time()
    model.fit(X_train_poly, y_train)
    training_time = time.time() - start_time

    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    results.append({"Degree":degree,"Train MSE":train_mse,"Test MSE":test_mse,"Train RMSE":train_rmse,"Test RMSE":test_rmse,"Train R2":train_r2,"Test R2":test_r2,"Training Time":training_time})

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(results_df["Degree"], results_df["Test RMSE"], marker="o")
plt.xlabel("Polynomial Degree")
plt.ylabel("Test RMSE")
plt.title("Effect of Polynomial Degree on Test RMSE")
plt.grid(True)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(results_df["Degree"], results_df["Train R2"], marker="o", label="Training R2")
plt.plot(results_df["Degree"], results_df["Test R2"], marker="o", label="Testing R2")
plt.xlabel("Polynomial Degree")
plt.ylabel("R2 Score")
plt.title("Training vs Testing R2")
plt.legend()
plt.grid(True)
plt.show()

best_degree = results_df.loc[results_df["Test RMSE"].idxmin(), "Degree"]
print("Best polynomial degree based on minimum Test RMSE:", int(best_degree))
print("As polynomial degree increases, model complexity increases.")
print("A very low degree may underfit the data, while a very high degree may overfit the data.")
print("This demonstrates the bias-variance trade-off.")